# EfficientNet-B4 Experiment 11 — TTA on Best Iter 10 Checkpoint

## Objective
Iteration 10 trained EfficientNet-B4 at 380×380 with metadata fusion and achieved Test F2 0.6611, AUC 0.9014. Despite higher resolution and more parameters, it underperformed B0 Iter 05 — diagnosed as over-regularisation (L1=1e-3 too aggressive for 17.5M params). This inference-only iteration applies 8× TTA to the Iter 10 checkpoint to test whether TTA can close the gap, following the same approach as Iter 06 (TTA on B0 Iter 05 pushed F2 from 0.6875 → 0.6952). No retraining; the Iter 10 checkpoint is loaded directly.

## Architecture

| Component | B4 Iter 10 (training) | B4 Iter 11 (TTA inference) |
|---|---|---|
| Model | `EfficientNetB4WithMetadata` | `EfficientNetB4WithMetadata` |
| Checkpoint | `efficientnet_b4_metadata_best.pth` | Same |
| Input resolution | 380×380 | 380×380 |
| Image directory | `data_new/images_380/` | `data_new/images_380/` (raw, transform=None) |
| Threshold tuning | `find_best_threshold` | **TTA predict loop on val set** |
| Test evaluation | `evaluate_model` | **TTA predict loop on test set** |
| TTA | No | **Yes (8×)** |

## Hypothesis
TTA averaging over 8 deterministic augmentations should reduce prediction variance on borderline cases, lowering the optimal threshold and boosting recall — the same mechanism that improved B0 (threshold 0.62 → 0.54, recall 0.8363 → 0.8830). For B4, the high threshold from Iter 10 (0.70) suggests the model is under-confident on positives; TTA averaging may bring probabilities into a more useful range and yield a more sensitive classifier.

## Import libraries, set seed, and choose device

In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from torchvision import transforms
from sklearn.metrics import fbeta_score

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
sys.path.insert(0, str(ROOT))

from src.data.dataset import HAM10000DatasetWithMetadata
from src.models.efficientnet import EfficientNetB4WithMetadata
from src.utils import seed_everything

seed_everything(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: cuda


## Load Datasets (no transform — TTA applies transforms manually)

In [2]:
METADATA_PATH      = str(ROOT / 'data_new/raw/HAM10000_metadata')
TEST_METADATA_PATH = str(ROOT / 'data_new/raw/ISIC2018_Task3_Test_GroundTruth.csv')

val_dataset_raw = HAM10000DatasetWithMetadata(
    csv_path=str(ROOT / 'data_new/splits/val.csv'),
    image_dir=str(ROOT / 'data_new/images_380/train'),
    metadata_path=METADATA_PATH,
    transform=None,
)

test_dataset_raw = HAM10000DatasetWithMetadata(
    csv_path=str(ROOT / 'data_new/splits/test.csv'),
    image_dir=str(ROOT / 'data_new/images_380/test'),
    metadata_path=TEST_METADATA_PATH,
    transform=None,
)

print(f'Val: {len(val_dataset_raw)} | Test: {len(test_dataset_raw)}')

Val: 2024 | Test: 1511


## Load Model from Iter 10 Checkpoint

In [3]:
METADATA_DIM = 17
DROPOUT      = 0.5

model = EfficientNetB4WithMetadata(
    metadata_dim=METADATA_DIM,
    num_classes=1,
    freeze_backbone=True,
    dropout=DROPOUT,
).to(device)

model.load_state_dict(torch.load(str(ROOT / 'models/efficientnet_b4_metadata_best.pth'), map_location=device))
model.eval()
print('Loaded Iter 10 checkpoint.')

Loaded Iter 10 checkpoint.


## TTA Setup

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _base(extra=None):
    ops = [transforms.Resize((380, 380))]
    if extra:
        ops += extra
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

tta_transforms = [
    _base(),                                                                      # 1. identity
    _base([transforms.RandomHorizontalFlip(p=1.0)]),                              # 2. H-flip
    _base([transforms.RandomVerticalFlip(p=1.0)]),                                # 3. V-flip
    _base([transforms.RandomHorizontalFlip(p=1.0),
           transforms.RandomVerticalFlip(p=1.0)]),                                # 4. HV-flip
    _base([transforms.RandomRotation(degrees=(90, 90))]),                         # 5. rotate 90
    _base([transforms.RandomRotation(degrees=(180, 180))]),                       # 6. rotate 180
    _base([transforms.RandomRotation(degrees=(270, 270))]),                       # 7. rotate 270
    _base([transforms.ColorJitter(brightness=0.1, contrast=0.1)]),                # 8. color jitter
]


def tta_predict(model, dataset, device, tta_transforms):
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for idx in range(len(dataset)):
            pil_img, metadata, label = dataset[idx]
            metadata = metadata.unsqueeze(0).to(device)

            preds = []
            for t in tta_transforms:
                tensor = t(pil_img).unsqueeze(0).to(device)
                prob = torch.sigmoid(model(tensor, metadata)).item()
                preds.append(prob)

            all_probs.append(np.mean(preds))
            all_labels.append(label)

    return np.array(all_probs), np.array(all_labels)


print(f'TTA: {len(tta_transforms)} augmentations per sample')

TTA: 8 augmentations per sample


## Threshold Tuning on Validation Set

In [5]:
print('Running TTA on validation set...')
val_probs, val_labels = tta_predict(model, val_dataset_raw, device, tta_transforms)

thresholds = np.arange(0.01, 0.90, 0.01)
f2_scores  = [fbeta_score(val_labels, (val_probs >= t).astype(int), beta=2, pos_label=1, zero_division=0)
              for t in thresholds]
best_threshold = thresholds[np.argmax(f2_scores)]
print(f'Best threshold: {best_threshold:.2f} | Val F2: {max(f2_scores):.4f}')

Running TTA on validation set...
Best threshold: 0.75 | Val F2: 0.6731


## Test Set Evaluation

In [6]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, classification_report

print('Running TTA on test set...')
test_probs, test_labels = tta_predict(model, test_dataset_raw, device, tta_transforms)
all_preds = (test_probs >= best_threshold).astype(int)

auc     = roc_auc_score(test_labels, test_probs)
bal_acc = balanced_accuracy_score(test_labels, all_preds)
f2      = fbeta_score(test_labels, all_preds, beta=2, pos_label=1, zero_division=0)

print(f'Threshold:          {best_threshold:.2f}')
print(f'AUC-ROC:            {auc:.4f}')
print(f'Balanced Accuracy:  {bal_acc:.4f}')
print(f'F2 Score:           {f2:.4f}')
print()
print(classification_report(test_labels, all_preds, target_names=['Non-Melanoma', 'Melanoma'], digits=4))

Running TTA on test set...
Threshold:          0.75
AUC-ROC:            0.9058
Balanced Accuracy:  0.8257
F2 Score:           0.6782

              precision    recall  f1-score   support

Non-Melanoma     0.9662    0.8970    0.9303      1340
    Melanoma     0.4831    0.7544    0.5890       171

    accuracy                         0.8809      1511
   macro avg     0.7247    0.8257    0.7597      1511
weighted avg     0.9116    0.8809    0.8917      1511

